# Hands-on: Sympy CSE application to BGK LBE

<div style="
    background-color: #f8f9fa; 
    border-left: 5px solid #ced4da; 
    padding: 15px; 
    margin: 10px 0; 
    border-radius: 4px; 
    color: #495057;
">
    <strong>Reference:</strong> Notebook can be found at <a href="https://github.com/hayashi-workshop/clbmtaichi/blob/main/generator/handson_derivation.ipynb">generator/handson_derivation.ipynb</a>
</div>

This hands-on notebook is designed to learn how we can use `sympy` to derive CSE-optimized collision kernel in `clbmtaichi`. Enjoy!

In [1]:
import sympy as sp
from sympy import symbols, Eq

## Weights

The weight vector $w_{i}$ has its intrinsic feature. If we label $w_{i}$ as $w_{\alpha \beta}$ = $w_{00}$, $w_{10}$, $w_{01}$, $\cdots$ for $\mathbf{c}=(0,0)$, $\mathbf{c}=(1,0)$, $\mathbf{c}=(0,1)$, $\cdots$, the velocity components $0$ and $\pm 1$ have their base weights as $2/3$ and $1/6$, respectively; for example, $w_{00} = \frac{2}{3} \times \frac{2}{3} = \frac{4}{9}$ and $w_{22} = \frac{1}{6} \times \frac{1}{6} = \frac{1}{36}$. Therefore, the weight vector is constructed by the following function. 

In [2]:
def calculate_weight(vec):
    w_base = {0: sp.Rational(2, 3), 1: sp.Rational(1, 6), -1: sp.Rational(1, 6)} # sp.Rational defines rational numbers
    weight = 1
    for component in vec:
        weight *= w_base[component]
    return weight

Let's check wether the function correctly work. 

In [3]:
w0 = calculate_weight( ( 0, 0) )
w1, w2, w3, w4 = calculate_weight( ( 1, 0) ), calculate_weight( ( 0, 1) ), calculate_weight( (-1, 0) ), calculate_weight( ( 0,-1) )
w5, w6, w7, w8 = calculate_weight( ( 1, 1) ), calculate_weight( (-1, 1) ), calculate_weight( (-1,-1) ), calculate_weight( ( 1,-1) )
print(f"w0, w1, w2, w3, w4, w5, w6, w7, w8 = {w0}, {w1}, {w2}, {w3}, {w4}, {w5}, {w6}, {w7}, {w8}")
print(f"sum w = {w0+w1+w2+w3+w4+w5+w6+w7+w8}")

w0, w1, w2, w3, w4, w5, w6, w7, w8 = 4/9, 1/9, 1/9, 1/9, 1/9, 1/36, 1/36, 1/36, 1/36
sum w = 1


Good!

## Velocity sets 

First, import required libraries before creating lattice velocity vectors. 

In [4]:
import itertools
import math
import numpy as np

The components of the D2Q9 velocity vectors take either $-1$, $0$ or $1$. D2Q9 consists of all possible combinations of them, and `itertools` easily produces such combinations, that is, 

In [5]:
dim = 2 # dimension
list(itertools.product([-1, 0, 1], repeat=dim)) # result: [(-1, -1), (-1, 0), (-1, 1), (0, -1), (0, 0), (0, 1), (1, -1), (1, 0), (1, 1)]

[(-1, -1), (-1, 0), (-1, 1), (0, -1), (0, 0), (0, 1), (1, -1), (1, 0), (1, 1)]

These are $\mathbf{c}$ we need! However, we expect the order $\mathbf{c}_{0} = (0,0)$, $\mathbf{c}_{1} = (1,0)$, $\cdots$, but the order of the velocities above does not follow this. The following function solves this by sorting the components: 

In [6]:
# Here we develop 2D version only, so dim=2 is not mandatory. However, we shall leave dim for later convinience. 
def create_vectors(dim):
    all_vectors = list(itertools.product([-1, 0, 1], repeat=dim))

    vectors = []

    def lbm_2d_sort_key(vec):
        length_sq = sum(c**2 for c in vec) # cx^2 + cy^2 
        
        if length_sq == 0: return (0, -100) # c0
            
        angle = math.atan2(vec[1], vec[0]) # angle between c and x-axis
        if angle < 0: angle += 2 * math.pi # quadrant correction

        return (length_sq, angle)

    # sort for (length_sq, angle). (0,0) is placed first since its length is 0. Next, four components of length_sq = 1 are sorted for angle. Then, ..
    sorted_pairs = sorted([(v, calculate_weight(v)) for v in all_vectors], key=lambda x: lbm_2d_sort_key(x[0]))

    # pick sorted vector from the result and list them. 
    vectors = [p[0] for p in sorted_pairs]

    return vectors

In [7]:
vectors = create_vectors(dim=dim)
print(vectors) # result : [(0, 0), (1, 0), (0, 1), (-1, 0), (0, -1), (1, 1), (-1, 1), (-1, -1), (1, -1)]

[(0, 0), (1, 0), (0, 1), (-1, 0), (0, -1), (1, 1), (-1, 1), (-1, -1), (1, -1)]


Thus, we are ready to create the weight vector: 

In [8]:
weights = [calculate_weight(v) for v in vectors]
print(weights)

[4/9, 1/9, 1/9, 1/9, 1/9, 1/36, 1/36, 1/36, 1/36]


## Macroscopic variables

We shall start using sympy to construct LBE! 

sympy *symbol* is a symbol variable to handle symbolic calculation. Deine *name* of symbol; then create the *symbol* for it by `sp.Symbol`. 

In [9]:
rho_name = 'rho'
rho = sp.Symbol(rho_name)

display(rho) # display renders symbols with LaTeX drawing

rho

Multiple symbols can be defined simultaneously from `List` using `sp.symbols`. 

In [10]:
vel_names = ['u', 'v', 'w'][:dim] # slicing eliminate 'w' when dim=2. 
vel_syms  = sp.symbols(vel_names)

display(sp.Matrix(vel_syms).T) # trick to display in LaTeX form. Not mandatory.

Matrix([[u, v]])

`rho = sp.Symbol('rho')`, of course, works. However, defining *name* sparately is sometimes useful. 

In [11]:
num_pops = len(vectors) # number of populations

f_syms = sp.symbols([f'f{i}' for i in range(num_pops)]) # sympy symbol of f

display(sp.Matrix(f_syms).T)

Matrix([[f0, f1, f2, f3, f4, f5, f6, f7, f8]])

What is the density in LBM? It is $\rho = \sum_{i=0}^{Q-1} f_{i}$. Get it!

In [12]:
rho_expr = sum(f_syms)

display(rho_expr)

f0 + f1 + f2 + f3 + f4 + f5 + f6 + f7 + f8

We've gotten into the discrete LB world. 

The term `expr` in `rho_expr` is an abbreviation of *expression*. The symbol $\rho$ is a symbol. `rho_expr` is a concrete *expression* of $\rho$. The name is arbitrary and you can use another one as you like. However, it would be better to have your own ruling for the variable names (my own ruling is sometimes broken in the actual code. sorry!). Let's have velocity expressions $\mathbf{u} = \sum \mathbf{c}_{i} f_{i} / \rho$ as well: 

In [13]:
vel_exprs = []
for d_idx in range(dim): 
    momentum_expr = sum(vec[d_idx] * f_syms[i] for i, vec in enumerate(vectors))
    vel_exprs.append( momentum_expr / ( rho_expr ) ) 

display(sp.Matrix(vel_exprs).T)

Matrix([[(f1 - f3 + f5 - f6 - f7 + f8)/(f0 + f1 + f2 + f3 + f4 + f5 + f6 + f7 + f8), (f2 - f4 + f5 + f6 - f7 - f8)/(f0 + f1 + f2 + f3 + f4 + f5 + f6 + f7 + f8)]])

## Equilibrium distribution function

Same as `f_syms`, we prepare `feq_syms` for $f_{i}^{eq}$. 

In [14]:
feq_syms = sp.symbols([f'feq{i}' for i in range(num_pops)])

display(sp.Matrix(feq_syms).T)

Matrix([[feq0, feq1, feq2, feq3, feq4, feq5, feq6, feq7, feq8]])

The most elaborative part in BGK is reconstruction of $f_{i}^{eq} = w_{i} \rho \left[ 1 + \frac{\mathbf{c}_{i} \cdot \mathbf{u}}{c_{s}^{2}} + \frac{(\mathbf{c}_{i} \cdot \mathbf{u})^{2}}{2 c_{s}^{4}} - \frac{\mathbf{u} \cdot \mathbf{u}}{2 c_{s}^{2}} \right]$. However, sympy enables us to quickly implement this as follows. 

In [15]:
cs2 = sp.Rational(1, 3) # cs^2 = 1/3

def create_feq_list(dim, rho, u, vectors, weights):
    u_sq_sum = sum(u_i**2 for u_i in u) # u^2 + v^2
    
    feq_list = []
    for idx, vec in enumerate(vectors):
        c_dot_u = sum(cx * u_i for cx, u_i in zip(vec, u)) # c_{i} dot u
    
        feq_list.append( weights[idx] * rho * (1 + c_dot_u/cs2 + (c_dot_u**2)/(2 * cs2**2) - u_sq_sum/(2 * cs2)) )

    return feq_list

$f^{eq}$ in *symbolic form*:

In [16]:
feq_raw_list = create_feq_list(dim, rho, vel_syms, vectors, weights)

display(sp.Matrix( feq_raw_list ).T)

Matrix([[4*rho*(-3*u**2/2 - 3*v**2/2 + 1)/9, rho*(3*u**2 + 3*u - 3*v**2/2 + 1)/9, rho*(-3*u**2/2 + 3*v**2 + 3*v + 1)/9, rho*(3*u**2 - 3*u - 3*v**2/2 + 1)/9, rho*(-3*u**2/2 + 3*v**2 - 3*v + 1)/9, rho*(-3*u**2/2 + 3*u - 3*v**2/2 + 3*v + 9*(u + v)**2/2 + 1)/36, rho*(-3*u**2/2 - 3*u - 3*v**2/2 + 3*v + 9*(-u + v)**2/2 + 1)/36, rho*(-3*u**2/2 - 3*u - 3*v**2/2 - 3*v + 9*(-u - v)**2/2 + 1)/36, rho*(-3*u**2/2 + 3*u - 3*v**2/2 - 3*v + 9*(u - v)**2/2 + 1)/36]])

Good; this is $f^{eq}$ that we know. However, $\rho$ and $\mathbf{u}$ are actually functions of $f$ in the discrete sense. So, by sending `expr` to the $f^{eq}$ create function, we obtain

In [17]:
feq_exprs = create_feq_list(dim, rho_expr, vel_exprs, vectors, weights)

display(feq_exprs[0])

(-3*(f1 - f3 + f5 - f6 - f7 + f8)**2/(2*(f0 + f1 + f2 + f3 + f4 + f5 + f6 + f7 + f8)**2) - 3*(f2 - f4 + f5 + f6 - f7 - f8)**2/(2*(f0 + f1 + f2 + f3 + f4 + f5 + f6 + f7 + f8)**2) + 1)*(4*f0/9 + 4*f1/9 + 4*f2/9 + 4*f3/9 + 4*f4/9 + 4*f5/9 + 4*f6/9 + 4*f7/9 + 4*f8/9)

Too complex! Don't worry, `sympy` is human friendly. `sp.simplify` helps us to grasp the structure of equation. 

In [18]:
sp.simplify(feq_exprs[0])

(-2*(f1 - f3 + f5 - f6 - f7 + f8)**2/3 - 2*(f2 - f4 + f5 + f6 - f7 - f8)**2/3 + 4*(f0 + f1 + f2 + f3 + f4 + f5 + f6 + f7 + f8)**2/9)/(f0 + f1 + f2 + f3 + f4 + f5 + f6 + f7 + f8)

## Lattice Boltzmann Equation

Prepare the relaxation rate. 

In [19]:
omega = sp.Symbol('omega')

display(omega)

omega

#### Symbolic LBE: *post-collision* distribution

We construct LBE and register them in Python *dict*. 

In [20]:
f_new_map = {
    f"f{i}": (f_syms[i] - omega * (f_syms[i] - feq_syms[i]))
    for i in range(num_pops)
}

In [21]:
f_new_map

{'f0': f0 - omega*(f0 - feq0),
 'f1': f1 - omega*(f1 - feq1),
 'f2': f2 - omega*(f2 - feq2),
 'f3': f3 - omega*(f3 - feq3),
 'f4': f4 - omega*(f4 - feq4),
 'f5': f5 - omega*(f5 - feq5),
 'f6': f6 - omega*(f6 - feq6),
 'f7': f7 - omega*(f7 - feq7),
 'f8': f8 - omega*(f8 - feq8)}

Each equation can be accessed via *key* of the dict, e.g., 

In [22]:
f_new_map['f0']

f0 - omega*(f0 - feq0)

## LBE: *standard* pipeline

We have, thus, constructed the $\rho$ equation, the $\mathbf{u}$ equations, the symbolic $f^{eq}$ equation, and the LBE. The pipeline of BGK can thus be 

In [23]:
display(Eq(rho, rho_expr))

Eq(rho, f0 + f1 + f2 + f3 + f4 + f5 + f6 + f7 + f8)

In [24]:
for d_idx in range(dim):
    display(Eq(vel_syms[d_idx], vel_exprs[d_idx]))

Eq(u, (f1 - f3 + f5 - f6 - f7 + f8)/(f0 + f1 + f2 + f3 + f4 + f5 + f6 + f7 + f8))

Eq(v, (f2 - f4 + f5 + f6 - f7 - f8)/(f0 + f1 + f2 + f3 + f4 + f5 + f6 + f7 + f8))

In [25]:
for i in range(num_pops):
    display( Eq(feq_syms[i], feq_raw_list[i]) )

Eq(feq0, 4*rho*(-3*u**2/2 - 3*v**2/2 + 1)/9)

Eq(feq1, rho*(3*u**2 + 3*u - 3*v**2/2 + 1)/9)

Eq(feq2, rho*(-3*u**2/2 + 3*v**2 + 3*v + 1)/9)

Eq(feq3, rho*(3*u**2 - 3*u - 3*v**2/2 + 1)/9)

Eq(feq4, rho*(-3*u**2/2 + 3*v**2 - 3*v + 1)/9)

Eq(feq5, rho*(-3*u**2/2 + 3*u - 3*v**2/2 + 3*v + 9*(u + v)**2/2 + 1)/36)

Eq(feq6, rho*(-3*u**2/2 - 3*u - 3*v**2/2 + 3*v + 9*(-u + v)**2/2 + 1)/36)

Eq(feq7, rho*(-3*u**2/2 - 3*u - 3*v**2/2 - 3*v + 9*(-u - v)**2/2 + 1)/36)

Eq(feq8, rho*(-3*u**2/2 + 3*u - 3*v**2/2 - 3*v + 9*(u - v)**2/2 + 1)/36)

In [26]:
for key, expr in zip(f_new_map.keys(), f_new_map.values()):
    display( Eq(sp.Symbol(key), expr) )

Eq(f0, f0 - omega*(f0 - feq0))

Eq(f1, f1 - omega*(f1 - feq1))

Eq(f2, f2 - omega*(f2 - feq2))

Eq(f3, f3 - omega*(f3 - feq3))

Eq(f4, f4 - omega*(f4 - feq4))

Eq(f5, f5 - omega*(f5 - feq5))

Eq(f6, f6 - omega*(f6 - feq6))

Eq(f7, f7 - omega*(f7 - feq7))

Eq(f8, f8 - omega*(f8 - feq8))

Compute the density and viscosites, compute the equilibrium distributions from those macro vars, and then do collision... There are no differences from the standard implementaion of BGK-LBE. We can obtain a BGK code by exporting this story. But, this is not our goal; our remaining task is to apply CSE to the LBE!

## Sympy Common Subexpression Elimination (CSE): application to LBE pipeline

Bundle the macroscopic variables. 

In [27]:
macro_keys   = [rho_name] + vel_names       # ['rho', 'u', 'v']
macro_values = [rho_expr] + vel_exprs[:dim] # [ f0 + f1 + ..., (f1 - f3 + ...)/(f0 + f1 + ...), (f2 - f4 + ...)/(f0 + f1 + ...)]

macro_exprs  = dict(zip(macro_keys, macro_values)) # combine name and expressions as dict

print(macro_exprs)

{'rho': f0 + f1 + f2 + f3 + f4 + f5 + f6 + f7 + f8, 'u': (f1 - f3 + f5 - f6 - f7 + f8)/(f0 + f1 + f2 + f3 + f4 + f5 + f6 + f7 + f8), 'v': (f2 - f4 + f5 + f6 - f7 - f8)/(f0 + f1 + f2 + f3 + f4 + f5 + f6 + f7 + f8)}


We list all expressions to be injected to CSE. `cse_targets` is a `list` of `sympy` expressions. We extract `expr` from `macro_exprs` and `f_new_map`. It should be noted that `f_new_map` itself is *symbolic* expression, therefore the symbols are *substituted* with the concrete expressions `feq_exprs` via `subs_map`. Actually, `list(f_new_map.values())` can be replaced with `feq_exprs`, but it may be convenient to handle them as *dict*, in order to memory the meaning of equations. 

In [28]:
# mapping feq0 -> (rho + 3*u...) for CSE
subs_map = {feq_syms[i]: feq_exprs[i] for i in range(num_pops)} 

# replacement of equations with feq_exprs
cse_targets = [ 
    expr.xreplace(subs_map) for expr in (list(macro_exprs.values()) + list(f_new_map.values()))
]

In [29]:
display(cse_targets[5]) # confirm LBE of f2 

f2 - omega*(f2 - (-3*(f1 - f3 + f5 - f6 - f7 + f8)**2/(2*(f0 + f1 + f2 + f3 + f4 + f5 + f6 + f7 + f8)**2) + 3*(f2 - f4 + f5 + f6 - f7 - f8)**2/(f0 + f1 + f2 + f3 + f4 + f5 + f6 + f7 + f8)**2 + 3*(f2 - f4 + f5 + f6 - f7 - f8)/(f0 + f1 + f2 + f3 + f4 + f5 + f6 + f7 + f8) + 1)*(f0/9 + f1/9 + f2/9 + f3/9 + f4/9 + f5/9 + f6/9 + f7/9 + f8/9))

CSE's turn!

In [30]:
replacements, reduced_exprs = sp.cse(cse_targets)

In [31]:
replacements 
# this is a list of 2-component touples
# (intermediate variables like x0, expr: expressions of intermediate variables like f1 + f8)
# 
# Eq(replacements[0][0], replacements[0][1]) -> x0 = f1 + f8

[(x0, f1 + f8),
 (x1, f2 + f6),
 (x2, f0 + f3 + f4 + f5 + f7 + x0 + x1),
 (x3, 1/x2),
 (x4, f5 - f7),
 (x5, -f3 - f6 + x0 + x4),
 (x6, x3*x5),
 (x7, -f4 - f8 + x1 + x4),
 (x8, x3*x7),
 (x9, x7**2),
 (x10, x2**(-2)),
 (x11, 3*x10/2),
 (x12, x11*x9),
 (x13, x5**2),
 (x14, x11*x13),
 (x15, x14 - 1),
 (x16, x12 + x15),
 (x17, f0/9 + f1/9 + f2/9 + f3/9 + f4/9 + f5/9 + f6/9 + f7/9 + f8/9),
 (x18, 3*x6),
 (x19, 3*x10),
 (x20, 1 - x12),
 (x21, x13*x19 + x20),
 (x22, 3*x8),
 (x23, -x14),
 (x24, x22 + x23),
 (x25, -x18),
 (x26, f0/36 + f1/36 + f2/36 + f3/36 + f4/36 + f5/36 + f6/36 + f7/36 + f8/36),
 (x27, x6 + x8),
 (x28, x20 + x24),
 (x29, x6 - x8)]

In [32]:
reduced_exprs 
# this is a list of reduces expression of *given expressions*

[x2,
 x6,
 x8,
 f0 - omega*(f0 + x16*(4*f0/9 + 4*f1/9 + 4*f2/9 + 4*f3/9 + 4*f4/9 + 4*f5/9 + 4*f6/9 + 4*f7/9 + 4*f8/9)),
 f1 - omega*(f1 - x17*(x18 + x21)),
 f2 - omega*(f2 - x17*(x19*x9 + x24 + 1)),
 f3 - omega*(f3 - x17*(x21 + x25)),
 f4 - omega*(f4 - x17*(3*x10*x9 - x15 - x22)),
 f5 - omega*(f5 - x26*(x18 + 9*x27**2/2 + x28)),
 f6 - omega*(f6 - x26*(x25 + x28 + 9*x29**2/2)),
 f7 - omega*(f7 - x26*(-x16 - x18 - x22 + 9*x27**2/2)),
 f8 - omega*(f8 - x26*(x18 + x20 - x22 + x23 + 9*x29**2/2))]

Note that the order of the components in `reduced_exprs` is the same as in `cse_targets`. Therefore, 

In [33]:
Eq(rho, reduced_exprs[0])

Eq(rho, x2)

Validity check!

In [34]:
x0 = replacements[0][1]
x1 = replacements[1][1]
x2 = replacements[2][1]

In [35]:
(x2.subs({'x1': x1})).subs({'x0': x0}) - rho_expr # x2 - rho = 0?
# subs is substitution operation

0

## Export as kernel

Finaly, we export the result as *collision-streaming kernel*. 

### Pull-type streaming

In [36]:
for i in range(num_pops):
    print(f"{f_syms[i]} = f_pre[I-c[{i}]][{i}]")

f0 = f_pre[I-c[0]][0]
f1 = f_pre[I-c[1]][1]
f2 = f_pre[I-c[2]][2]
f3 = f_pre[I-c[3]][3]
f4 = f_pre[I-c[4]][4]
f5 = f_pre[I-c[5]][5]
f6 = f_pre[I-c[6]][6]
f7 = f_pre[I-c[7]][7]
f8 = f_pre[I-c[8]][8]


### Construction of $f^{eq}$

In [37]:
for var, expr in replacements:
    print(f"{var} = {expr}")

x0 = f1 + f8
x1 = f2 + f6
x2 = f0 + f3 + f4 + f5 + f7 + x0 + x1
x3 = 1/x2
x4 = f5 - f7
x5 = -f3 - f6 + x0 + x4
x6 = x3*x5
x7 = -f4 - f8 + x1 + x4
x8 = x3*x7
x9 = x7**2
x10 = x2**(-2)
x11 = 3*x10/2
x12 = x11*x9
x13 = x5**2
x14 = x11*x13
x15 = x14 - 1
x16 = x12 + x15
x17 = f0/9 + f1/9 + f2/9 + f3/9 + f4/9 + f5/9 + f6/9 + f7/9 + f8/9
x18 = 3*x6
x19 = 3*x10
x20 = 1 - x12
x21 = x13*x19 + x20
x22 = 3*x8
x23 = -x14
x24 = x22 + x23
x25 = -x18
x26 = f0/36 + f1/36 + f2/36 + f3/36 + f4/36 + f5/36 + f6/36 + f7/36 + f8/36
x27 = x6 + x8
x28 = x20 + x24
x29 = x6 - x8


### Collision and writing

In [38]:
for i in range(num_pops):
    q = i+1+dim
    print(f"f_post[I][{i}] = {reduced_exprs[q]}")

f_post[I][0] = f0 - omega*(f0 + x16*(4*f0/9 + 4*f1/9 + 4*f2/9 + 4*f3/9 + 4*f4/9 + 4*f5/9 + 4*f6/9 + 4*f7/9 + 4*f8/9))
f_post[I][1] = f1 - omega*(f1 - x17*(x18 + x21))
f_post[I][2] = f2 - omega*(f2 - x17*(x19*x9 + x24 + 1))
f_post[I][3] = f3 - omega*(f3 - x17*(x21 + x25))
f_post[I][4] = f4 - omega*(f4 - x17*(3*x10*x9 - x15 - x22))
f_post[I][5] = f5 - omega*(f5 - x26*(x18 + 9*x27**2/2 + x28))
f_post[I][6] = f6 - omega*(f6 - x26*(x25 + x28 + 9*x29**2/2))
f_post[I][7] = f7 - omega*(f7 - x26*(-x16 - x18 - x22 + 9*x27**2/2))
f_post[I][8] = f8 - omega*(f8 - x26*(x18 + x20 - x22 + x23 + 9*x29**2/2))


### Macroscopic variables

In [39]:
print(f"{rho}[I] = {reduced_exprs[0]}")

for d_idx in range(dim):
    print(f"vel[I][{d_idx}] = {reduced_exprs[1+d_idx]}")

rho[I] = x2
vel[I][0] = x6
vel[I][1] = x8


## Functionalize expressions

In [40]:
idt = ['', '    ', '        ']

print(f"{idt[0]}@ti.kernel")
print(f"{idt[0]}def col_stream_core(rho: ti.template(), vel: ti.template(), f_pre: ti.template(), f_post: ti.template()):")

print(f"{idt[1]}for I in ti.grouped(rho):")

for i in range(num_pops):
    print(f"{idt[2]}{f_syms[i]} = f_pre[I-c[{i}]][{i}]")

for var, expr in replacements:
    print(f"{idt[2]}{var} = {expr}")

for i in range(num_pops):
    q = i+1+dim
    print(f"{idt[2]}f_post[I][{i}] = {reduced_exprs[q]}")

print(f"{idt[2]}{rho}[I] = {reduced_exprs[0]}")

for d_idx in range(dim):
    print(f"{idt[2]}vel[I][{d_idx}] = {reduced_exprs[1+d_idx]}")

@ti.kernel
def col_stream_core(rho: ti.template(), vel: ti.template(), f_pre: ti.template(), f_post: ti.template()):
    for I in ti.grouped(rho):
        f0 = f_pre[I-c[0]][0]
        f1 = f_pre[I-c[1]][1]
        f2 = f_pre[I-c[2]][2]
        f3 = f_pre[I-c[3]][3]
        f4 = f_pre[I-c[4]][4]
        f5 = f_pre[I-c[5]][5]
        f6 = f_pre[I-c[6]][6]
        f7 = f_pre[I-c[7]][7]
        f8 = f_pre[I-c[8]][8]
        x0 = f1 + f8
        x1 = f2 + f6
        x2 = f0 + f3 + f4 + f5 + f7 + x0 + x1
        x3 = 1/x2
        x4 = f5 - f7
        x5 = -f3 - f6 + x0 + x4
        x6 = x3*x5
        x7 = -f4 - f8 + x1 + x4
        x8 = x3*x7
        x9 = x7**2
        x10 = x2**(-2)
        x11 = 3*x10/2
        x12 = x11*x9
        x13 = x5**2
        x14 = x11*x13
        x15 = x14 - 1
        x16 = x12 + x15
        x17 = f0/9 + f1/9 + f2/9 + f3/9 + f4/9 + f5/9 + f6/9 + f7/9 + f8/9
        x18 = 3*x6
        x19 = 3*x10
        x20 = 1 - x12
        x21 = x13*x19 + x20
        x22 = 3*x

Congratulation! Compare this result with `lb_solver/d2q9_BGK_kernel.py`. Except some additional functions in the latter, the derived kernels are the same. 

Sympy CSE is now under control in your hands. 

Good news. Just by extending the `vectors` to D3Q29, we would immediately derive 3D kernerl as well!